# Phase 2b Stage 2: Entity聚类分析

将Stage 1.5重分配后的626个entities聚类到200-250个标准entities

## 方法：
- 使用BERTopic（自适应聚类数，基于HDBSCAN）
- 对每个relation独立处理
- 按频次选择canonical name

In [ ]:
# 初始化聚类器
clusterer = EntityClusterer(
    stage1_5_result_path='../results/entity_redistribution_stage1_5_redistributed.json',
    phase1_result_path='../results/phase1_5percent_exploration.json'  # 用于获取entity频率
)

# 加载Stage 1.5数据
stage1_5_data = clusterer.load_stage1_5_results()

# 加载Phase 1数据和relation mapping（获取entity频率）
clusterer.load_phase1_and_mapping(
    relation_mapping_path='../results/relation_mapping_final_v2.json'
)

# 提取entities（按relation分组）
entities_by_relation = clusterer.extract_entities_by_relation(stage1_5_data)

## 1. 初始化并加载Stage 1.5数据

In [ ]:
# 初始化聚类器
clusterer = EntityClusterer(
    stage1_5_result_path='../results/entity_redistribution_stage1_5_redistributed.json'
)

# 加载数据
stage1_5_data = clusterer.load_stage1_5_results()

In [ ]:
# 提取entities（按relation分组）
entities_by_relation = clusterer.extract_entities_by_relation(stage1_5_data)

## 2. 测试单个relation（visual_theme）

In [ ]:
# 选择一个有代表性的relation进行测试
TEST_RELATION = 'visual_theme'

print(f"测试relation: {TEST_RELATION}")
print(f"Unique entities: {len(clusterer.entity_counts_by_relation[TEST_RELATION])}")

In [ ]:
# 生成BGE embeddings
embeddings, unique_entities = clusterer.embed_entities_bge(
    relation=TEST_RELATION,
    model_name="BAAI/bge-base-en-v1.5"
)

In [ ]:
# BERTopic聚类
topic_model, topics, probs = clusterer.cluster_with_bertopic(
    relation=TEST_RELATION,
    embeddings=embeddings,
    unique_entities=unique_entities,
    min_cluster_size=2,
    verbose=True
)

In [ ]:
# 查看每个topic的详细entities
print(f"\nDetailed topic breakdown for {TEST_RELATION}:")
print("="*80)

entity_counter = clusterer.entity_counts_by_relation[TEST_RELATION]

for topic_id in sorted(set(topics)):
    topic_entities = [
        (unique_entities[i], entity_counter[unique_entities[i]])
        for i, t in enumerate(topics)
        if t == topic_id
    ]
    topic_entities.sort(key=lambda x: x[1], reverse=True)
    
    total_count = sum(count for _, count in topic_entities)
    
    if topic_id == -1:
        print(f"\nTopic {topic_id} (Noise): {len(topic_entities)} entities ({total_count} instances)")
    else:
        print(f"\nTopic {topic_id}: {len(topic_entities)} entities ({total_count} instances)")
    
    for entity, count in topic_entities:
        marker = "★" if count > 5 else " "
        print(f"  {marker} {entity:40s}: {count:3d}")

In [ ]:
# 生成entity mapping
entity_mapping_test = clusterer.generate_entity_mapping(
    relation=TEST_RELATION,
    topics=topics,
    unique_entities=unique_entities,
    method='frequency'
)

In [ ]:
# 查看mapping详情
print("\nEntity Mapping Details:")
print("="*80)

# 按canonical name分组显示
canonical_groups = {}
for orig, canonical in entity_mapping_test.items():
    if canonical not in canonical_groups:
        canonical_groups[canonical] = []
    canonical_groups[canonical].append(orig)

for canonical in sorted(canonical_groups.keys()):
    group = canonical_groups[canonical]
    if len(group) == 1:
        # 单个entity，未合并
        continue
    
    total_count = sum(entity_counter[ent] for ent in group)
    print(f"\n{canonical} ({total_count} instances):")
    for ent in sorted(group):
        if ent == canonical:
            print(f"  ★ {ent:40s}: {entity_counter[ent]:3d} [CANONICAL]")
        else:
            print(f"    {ent:40s}: {entity_counter[ent]:3d}")

## 3. 处理所有relations

In [ ]:
# 如果测试结果满意，处理所有relations
all_entity_mappings = {}

for idx, relation in enumerate(sorted(clusterer.entity_counts_by_relation.keys()), 1):
    print(f"\n{'='*80}")
    print(f"[{idx}/15] Processing relation: {relation}")
    print(f"{'='*80}")
    
    n_entities = len(clusterer.entity_counts_by_relation[relation])
    
    # 如果entity数量太少（<=2），跳过聚类
    if n_entities <= 2:
        print(f"⚠️  Skipping '{relation}' (only {n_entities} entities, no clustering needed)")
        # 直接创建identity mapping
        all_entity_mappings[relation] = {
            ent: ent
            for ent in clusterer.entity_counts_by_relation[relation].keys()
        }
        continue
    
    # 生成embeddings
    embeddings, unique_entities = clusterer.embed_entities_bge(
        relation=relation,
        model_name="BAAI/bge-base-en-v1.5"
    )
    
    # BERTopic聚类
    topic_model, topics, probs = clusterer.cluster_with_bertopic(
        relation=relation,
        embeddings=embeddings,
        unique_entities=unique_entities,
        min_cluster_size=2,
        verbose=False
    )
    
    # 生成mapping
    entity_mapping = clusterer.generate_entity_mapping(
        relation=relation,
        topics=topics,
        unique_entities=unique_entities,
        method='frequency'
    )
    
    all_entity_mappings[relation] = entity_mapping

## 4. 最终统计

In [ ]:
print(f"\n{'='*80}")
print("Final Statistics")
print(f"{'='*80}")

for relation in sorted(all_entity_mappings.keys()):
    mapping = all_entity_mappings[relation]
    n_before = len(mapping)
    n_after = len(set(mapping.values()))
    compression = n_after / n_before if n_before > 0 else 1.0
    
    print(f"  {relation:25s}: {n_before:3d} → {n_after:3d} "
          f"(compression: {compression:.2%})")

total_before = sum(len(m) for m in all_entity_mappings.values())
total_after = sum(len(set(m.values())) for m in all_entity_mappings.values())

print(f"\n  {'TOTAL':25s}: {total_before:3d} → {total_after:3d} "
      f"(compression: {total_after/total_before:.2%})")

## 5. 保存结果

In [ ]:
# 保存entity mapping
clusterer.save_results(
    all_entity_mappings=all_entity_mappings,
    output_path='../results/entity_mapping_bertopic.json',
    method='bertopic',
    metadata={
        'embedding_model': 'BAAI/bge-base-en-v1.5',
        'min_cluster_size': 2,
        'selection_method': 'frequency'
    }
)

print("\n✅ Phase 2b Stage 2 完成！")
print(f"   Entity mapping已保存: results/entity_mapping_bertopic.json")
print(f"\n下一步: 如果需要LLM微调，参考refine_relation_mapping.py创建entity版本")